# Delft3D to Sandplover DataCube Conversion

This notebook converts a **Delft3D** hydrodynamic model output (NetCDF trim file) into a **[Sandplover](https://github.com/sandpiper-toolchain/sandplover) DataCube** — a structured 3D array format with dimensions `(time, x, y)` designed for coastal and deltaic model analysis.

## What this does
The converter reads a Delft3D `trim-*.nc` file, extracts and computes the following variables, and saves them as a Sandplover-compatible NetCDF DataCube:

| Output Variable | Source | Description |
|----------------|--------|-------------|
| `eta` | `S1 - DPS` | Bed elevation (water level minus water depth) |
| `water_depth` | `DPS` | Water depth from bed to surface |
| `velocity` | `sqrt(U1² + V1²)` | Velocity magnitude at surface layer |
| `mud_frac` | `MUDFRAC` | Mud fraction (0–1) |
| `sand_frac` | `1 - MUDFRAC` | Sand fraction (0–1) |

## Prerequisites
- `sandplover` must be installed: `pip install sandplover`
- `xarray`, `numpy`, `pandas` must be installed
- `delft3d_converter.py` must be in the same directory as this notebook

## Notes
- **x and y dimensions are swapped** in the output file relative to the Delft3D grid convention.
- To add more variables (bed shear stress, sediment transport, etc.), see **Section 3** below.

## 1. Import Converter

Import `Delft3DConverter` from `delft3d_converter.py`. This class handles the full pipeline:
loading the Delft3D NetCDF file → extracting variables → building a Sandplover DataCube → saving to NetCDF.

In [7]:
from delft3d_converter import Delft3DConverter

## 2. Run Conversion

Set `trim_file` to the path of your Delft3D NetCDF output (typically named `trim-*.nc`), and `output_file` to the desired path for the resulting DataCube.

**Parameters for `.convert()`:**
- `output_file` — path where the output DataCube NetCDF will be saved
- `show_stats=True` — prints shape, value range, mean, and std for each variable after conversion

In [8]:
# Set file paths: change to NETCDF file location
trim_file = 'trim-Test.nc'
output_file = 'sandplover_cube.nc'

# Run conversion
converter = Delft3DConverter(trim_file).convert(output_file, show_stats=True)

File loaded: trim-Test.nc
Bed elevation (eta = S1 - DPS): [-6.3145, 10.0000] m
Velocity magnitude: [0.0000, 1.1809] m/s
Sediment fractions calculated
Variables: ['eta', 'water_depth', 'velocity', 'mud_frac', 'sand_frac']
Dimensions: time=3, y=302, x=227
DataCube created: (3, 302, 227) (time, y, x)
   Variables: ['eta', 'water_depth', 'velocity', 'mud_frac', 'sand_frac']
Saved: sandplover_cube.nc
Variable Statistics:
--------------------------------------------------
eta:
  Shape: (3, 302, 227)
  Range: [-6.3145, 10.0000]
  Mean: -0.2908, Std: 3.4756
water_depth:
  Shape: (3, 302, 227)
  Range: [-5.0000, 7.4782]
  Mean: 1.3436, Std: 2.3855
velocity:
  Shape: (3, 302, 227)
  Range: [0.0000, 1.1809]
  Mean: 0.0751, Std: 0.1439
mud_frac:
  Shape: (3, 302, 227)
  Range: [0.0000, 1.0000]
  Mean: 0.4201, Std: 0.4071
sand_frac:
  Shape: (3, 302, 227)
  Range: [0.0000, 1.0000]
  Mean: 0.5799, Std: 0.4071


## 3. Additional Variables

By default, only `eta`, `water_depth`, `velocity`, `mud_frac`, and `sand_frac` are extracted.
To include more variables, open `delft3d_converter.py` and uncomment the relevant sections:

**Step 1 — Add the variable to `VARIABLE_MAPPING`** (top of the file):
```python
'TAUKSI': 'bed_shear_stress_x',   # Bed shear stress in x-direction
'TAUETA': 'bed_shear_stress_y',   # Bed shear stress in y-direction
```

**Step 2 — Uncomment the extraction logic in `extract_variables()`:**
```python
elif delft_var == 'TAUKSI' and len(var_data.shape) == 3:
    tau_x = var_data
```

**Step 3 — Uncomment the derived variable calculation:**
```python
if tau_x is not None and tau_y is not None:
    bed_shear_stress_magnitude = np.sqrt(tau_x**2 + tau_y**2)
    self.data_dict['bed_shear_stress'] = bed_shear_stress_magnitude
```

### Available optional variables:
| Variable | Delft3D key | Description |
|----------|-------------|-------------|
| Bed shear stress | `TAUKSI`, `TAUETA` | Shear stress components at the bed |
| Bed load transport | `SBUU`, `SBVV` | Bed load transport in x/y directions |
| Suspended load | `SSUU`, `SSVV` | Suspended sediment transport in x/y directions |
| Velocity direction | derived from `U1`, `V1` | Flow direction angle in degrees |